**2. MOCO TRAINING**

In [15]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

Progetto: floods


**SETUP PARAMETERS**

In [16]:
# Parametri job
job_name = "moco_v6"                                  # nome da passare poi a notebook moco 
dataset = "Test"    
epochs = 10                                # Test, Standard, Anomalies*
weights_encoder_sar = "train_s1_v3_Standard_200"     
weights_encoder_opt = "train_s2_v3_Standard_200"                
#handler = pretrain_encoders                        

parametri = {
    "epochs": epochs, 
    "batch_size": 16,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "momentum": 0.9,
    "patch_size": 256,
    "n_images1": 4, "n_channels1": 2,               # sar
    "n_images2": 4, "n_channels2": 10,              # ottico
    "moco_dim": 128,
    "moco_k": 64,                                 # dimensioni coda (multiplo batch_size),  standard: 1554, test: 64   
    "moco_t": 0.07,
    "symmetric": False,
    "mamba": False,
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "weights_encoder_sar": weights_encoder_sar,
    "weights_encoder_opt": weights_encoder_opt,
    "patience": 20,
    "min_delta": 1e-4,
    "time_debug": True                             # time_debug = True solo per debug, = False per training  
}

print(f"PARAMETRI: {parametri}")

# volume
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]



PARAMETRI: {'epochs': 10, 'batch_size': 16, 'lr': 0.001, 'weight_decay': 0.0001, 'momentum': 0.9, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'moco_dim': 128, 'moco_k': 64, 'moco_t': 0.07, 'symmetric': False, 'mamba': False, 'workers': 0, 'job_name': 'moco_v6', 'dataset': 'Test', 'weights_encoder_sar': 'train_s1_v3_Standard_200', 'weights_encoder_opt': 'train_s2_v3_Standard_200', 'patience': 20, 'min_delta': 0.0001, 'time_debug': True}


**BUILD ENVIRONMENT**

In [17]:
moco_func = project.new_function(
    name= f'moco_Floods_{dataset}_{job_name}_{epochs}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_moco", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "digitalhub==0.15.6", "digitalhub-runtime-python==0.15.2"]
)



# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = moco_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

2026-09-15 08:16:25,476 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:30,482 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:35,491 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:40,499 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:45,508 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:50,516 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08:16:55,525 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7c7ce6025679463c89ffad67b6d47ca3 to finish...
2026-09-15 08

BUILD: COMPLETED


**TRAINING**

In [18]:
# action job = avvia container, esegue script, libera risorse

run_moco = moco_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100",
    # local_execution= True,                       # 1x = 1 gpu
    wait=True
)

print(f"Run moco avviato: {run_moco.id}")
print(run_moco.status.state)
print(run_moco.status.message)

2026-09-15 08:18:16,280 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:21,286 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:26,296 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:31,305 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:36,315 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:41,324 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08:18:46,333 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 3f8b63fb0d1d429f8b96e72cae02a528 to finish...
2026-09-15 08

Run moco avviato: 3f8b63fb0d1d429f8b96e72cae02a528
COMPLETED
None


**PLOTS**

In [19]:
# salvataggio log
path_moco = project.get_artifact(f"moco-metrics_{job_name}_{dataset}_{epochs}").download(overwrite=True)
df_moco = pd.read_csv(path_moco)

# plot
plt.figure(figsize=(8, 5))
plt.plot(df_moco['epoch'], df_moco['train_loss'], color='green', label='Train Loss')
plt.title(f'Training MoCo - {job_name}_{dataset}_{epochs}')
plt.xlabel('Epochs')
plt.ylabel('Loss (InfoNCE)')
plt.grid(True)
plt.legend()
plt.show()

BackendError: No object found.